# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [1]:
import numpy as np 
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use. 

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text. 

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available. 

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [2]:
# news = pd.read_csv('/kaggle/input/topic-labeled-news-dataset/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

news = pd.read_csv('datasets//bbc_news.csv')
MAX_NEWS = 1000
DOCUMENT="description"
TOPIC="title"

# news = pd.read_csv('/kaggle/input/mit-ai-news-published-till-2023/articles.csv')
# MAX_NEWS = 100
# DOCUMENT="Article Body"
# TOPIC="Article Header"

# news = "PICK A DATASET" #Ideally pick one from the commented ones above

ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [3]:
news["id"] = news.index
news.head()

,title,pubDate,guid,link,description,id
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...,0
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as...",1
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...,2
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...,3
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...,4


In [4]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

## Install dependencies
Install the required packages for the modern LangChain stack:
- `langchain-core`: Core abstractions and LCEL (LangChain Expression Language)
- `langchain-chroma`: Official ChromaDB integration for LangChain
- `langchain-huggingface`: Official HuggingFace integration for LangChain
- `chromadb`: The vector database backend

In [5]:
%pip install chromadb langchain-core langchain-chroma langchain-huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import and configure the Vector Database
We use `langchain-chroma` which wraps ChromaDB with the LangChain `VectorStore` interface.

The `HuggingFaceEmbeddings` class from `langchain-huggingface` generates embeddings locally using a sentence-transformer model. This replaces the default ChromaDB embedding function and gives us full LangChain compatibility.

We use `persist_directory` to save the database to disk, keeping the same behaviour as before.

In [6]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model (sentence-transformers is lightweight and runs locally)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\jesur\Desktop\ACurso\w7\v.torch\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\jesur\Desktop\ACurso\w7\v.torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Filling and Querying the ChromaDB Database
With `langchain-chroma` we use `Chroma.from_texts()` to create the collection and populate it in one step.

Each document is stored with its text content and associated metadata (the topic). The collection is persisted to the specified directory, replacing any previous data.

In [7]:
import shutil
import os

PERSIST_DIR = "/path/to/persist/directory"
COLLECTION_NAME = "news_collection"

# Remove existing collection data to start fresh
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

# Create the vector store and populate it from the DataFrame in one step
vectorstore = Chroma.from_texts(
    texts=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)

Once the data is in the vector store, we can query it using `similarity_search()`. The search is semantic — it does not look for exact words or phrases, but for documents whose meaning is closest to the query.

The metadata is not used in the search, but can be accessed in the returned `Document` objects for filtering or display.

In [8]:
# similarity_search returns a list of LangChain Document objects
results = vectorstore.similarity_search(query="laptop", k=10)

print(results)

[Document(id='id775', metadata={'title': 'Laptop art: From Vans to Harry Styles'}, page_content='Photography student Thorsten Mjölnir captures the way students decorate their laptops.'), Document(id='id707', metadata={'title': "Not smart but clever? The return of 'dumbphones'"}, page_content='Why sales of very basic mobile phones, without apps and internet connection, are increasing.'), Document(id='id310', metadata={'title': 'Building a bigger home for the British Library collection'}, page_content="What do you do when your collection of millions of books keeps growing but your bookshelves don't?"), Document(id='id587', metadata={'title': 'How a jetpack design helped create a flying motorbike'}, page_content='The developers of a powerful mini aircraft hope it will be used by the armed forces.'), Document(id='id444', metadata={'title': 'The popular apps aiming to tame the chaos of family life'}, page_content='How tech is helping young families and couples regain their busy social lives

## Vector MAP

In [9]:
!pip install matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [11]:
# With langchain-chroma we access the underlying ChromaDB collection via the private attribute
raw_collection = vectorstore._collection

getado = raw_collection.get(
    ids=["id141"],
    include=["documents", "embeddings"]
)

In [12]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

array([[ 8.36052671e-02,  9.88162681e-03, -6.33521527e-02,
         6.83540199e-03,  9.83688086e-02,  6.62363693e-02,
         8.81885067e-02,  4.32013627e-03,  4.62585725e-02,
        -2.53718253e-02,  3.03160697e-02,  2.90436726e-02,
         3.73521782e-02,  1.74557716e-02, -3.08766887e-02,
        -4.47611697e-02, -7.60662183e-03, -5.80879189e-02,
        -4.53988723e-02, -6.97018504e-02, -3.50679597e-03,
         4.58132997e-02,  1.58200655e-02,  3.44685353e-02,
        -6.20313846e-02,  9.38327238e-02,  5.95623329e-02,
        -3.07523962e-02, -8.41464549e-02, -2.55898554e-02,
        -5.44907618e-03, -1.64661780e-02, -1.00670829e-01,
         5.22239953e-02, -6.52279612e-03,  6.44801930e-02,
         6.21989705e-02, -4.69200779e-03, -3.38836387e-02,
        -5.86140202e-04, -9.11897495e-02, -9.24299881e-02,
        -4.39610183e-02, -4.71336842e-02,  1.49749219e-03,
        -9.09673143e-03,  2.79689208e-02, -1.05015874e-01,
         6.03262298e-02,  5.91969714e-02, -2.59885825e-0

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents. 

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search. 


## Loading the model and creating the RAG chain
TRANSFORMERS + LCEL!!

We now use the modern **LangChain Expression Language (LCEL)** to build the RAG pipeline with the `|` pipe operator.

We are importing:
* **HuggingFacePipeline** from `langchain_huggingface`: wraps a `transformers.pipeline` as a LangChain-compatible LLM.
* **PromptTemplate** from `langchain_core.prompts`: creates the prompt with `{context}` and `{question}` placeholders.
* **StrOutputParser** from `langchain_core.output_parsers`: extracts the plain text string from the model output.
* **RunnablePassthrough** from `langchain_core.runnables`: passes the input unchanged through the chain.

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It has 3 billion parameters, more than enough for our sample, and works much better than GPT2.

Please feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending). My recommendation is to choose "small" models, or we will run out of memory in Kaggle.

In [13]:
# # Uninstall current version
# %pip uninstall transformers -y

# # Install the exact version recommended by the model card
# %pip install transformers==4.49.0

# # Also install the other recommended dependencies
# %pip install torch==2.6.0 accelerate==1.3.0 flash_attn==2.7.4.post1 --no-build-isolation

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "microsoft/Phi-4-mini-instruct"   # or "HuggingFaceTB/SmolLM3-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

# Then create your pipeline as usual


c:\Users\jesur\Desktop\ACurso\w7\v.torch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jesur\.cache\huggingface\hub\models--microsoft--Phi-4-mini-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  7.53s/it]
Some parameters are on th

The next step is to initialize the `transformers` pipeline and wrap it with `HuggingFacePipeline` so it becomes a LangChain-compatible LLM.

The model's response is limited to 256 tokens. Setting `device_map` to `auto` lets the model automatically choose the best available device (CPU or GPU).

In [17]:
from langchain_huggingface import HuggingFacePipeline

# Build the transformers pipeline as before
hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

# Wrap it as a LangChain LLM
llm = HuggingFacePipeline(pipeline=hf_pipe)

Device set to use cuda:0


## Building the RAG chain with LCEL

Instead of manually building the prompt string, we now use `PromptTemplate` with named placeholders (`{context}` and `{question}`).

The full RAG chain is assembled using the LCEL pipe operator `|`:
1. The **retriever** fetches the top-k relevant documents from ChromaDB.
2. The **prompt** template formats the context and the user question.
3. The **LLM** generates an answer based on the formatted prompt.
4. The **output parser** extracts the final text string.

You can limit the length of the context passed to the model to avoid memory issues with datasets that contain very long texts.

In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Define the prompt template with named placeholders
prompt_template = PromptTemplate.from_template(
    "Relevant context: {context}\n\nThe user's question: {question}"
)

# Create the retriever from the vector store (fetches top-10 similar documents)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# Helper to format a list of Document objects into a single context string
def format_docs(docs):
    context = " ".join([f"#{doc.page_content}" for doc in docs])
    # Optionally limit context length to avoid memory issues:
    # context = context[:5120]
    return context

# Build the LCEL RAG chain using the pipe operator
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print(prompt_template)

input_variables=['context', 'question'] input_types={} partial_variables={} template="Relevant context: {context}\n\nThe user's question: {question}"


Now all that remains is to invoke the chain with the user's question and wait for its response!


In [19]:
question = "Can I buy a Toshiba laptop?"

# invoke() replaces the manual pipe() call — the chain handles retrieval and generation end-to-end
response = rag_chain.invoke(question)

print(response)

Relevant context: #Photography student Thorsten Mjölnir captures the way students decorate their laptops. #Why sales of very basic mobile phones, without apps and internet connection, are increasing. #Omar Mehtab spoke to a 'scalper;' a person who stockpiles popular products and resells them for profit. #Animated film Jujutsu Kaisen 0 is a surprise box office hit, as anime enjoys a new global popularity. #What do you do when your collection of millions of books keeps growing but your bookshelves don't? #Many Russians who can afford it have fled to nearby countries since Russia invaded Ukraine. #Some Chinese cities are under a lockdown and the country could pay the price, analysts say. #The developers of a powerful mini aircraft hope it will be used by the armed forces. #Paper £20 and £50 notes cannot be used after September this year but can still be exchanged. #A buyer hasn't been found for the ailing supplier, so the the taxpayer will have to keep funding it.

The user's question: Ca